# AdaptiRank — M3 Three-Split Dense Handoff (Colab GPU)
Regenerates dense embeddings + FAISS index **once**, retrieves **train+validation+test**, and **durably saves the index to Drive** so it never has to be rebuilt again.

**Runtime -> Change runtime type -> A100 (or L4)** for the fastest ~1.2M-doc encode. `adaptirank_dataset.tar` should already be at `MyDrive/adaptirank/` from the earlier run.

## 1. Confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 2. Clone repo at the M3-handoff commit

In [ ]:
import subprocess, os
OWNER, REPO, COMMIT = 'YashDThapliyal', 'adaptirank', 'efe43a572b11bfd3071753001cfa346c2496e173'
subprocess.run(['git','clone',f'https://github.com/{OWNER}/{REPO}.git','/content/adaptirank'], check=True)
os.chdir('/content/adaptirank'); subprocess.run(['git','checkout',COMMIT], check=True)
print('HEAD:', subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True).stdout.strip())
print('dirty:', bool(subprocess.run(['git','status','--porcelain'],capture_output=True,text=True).stdout.strip()))

## 3. Install dependencies

In [ ]:
!pip -q install -e . 2>&1 | tail -5
import torch; print('cuda:', torch.cuda.is_available())

## 4. Mount Drive + place dataset

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import subprocess, os
FP='dda38161938e829f2c2fc9b73d40d6cf922a5470c3b45bf176f742ee0ca7c667'
dst='artifacts/datasets/esci/processed'; os.makedirs(dst, exist_ok=True)
subprocess.run(['tar','-xf','/content/drive/MyDrive/adaptirank/adaptirank_dataset.tar','-C',dst], check=True)
assert os.path.isfile(f'{dst}/{FP}/catalog.parquet'); print('dataset ready')

## 5. Dense retrieval for ALL THREE splits (CUDA)

In [ ]:
import subprocess, sys, time
t0=time.time()
p=subprocess.run([sys.executable,'scripts/run_retrieval.py','--config','configs/retrieval/m3_three_split.yaml','--method','dense'],capture_output=True,text=True)
print('elapsed_min', round((time.time()-t0)/60,1))
print(chr(10).join(p.stdout.splitlines()[-8:])); print('--STDERR--'); print(chr(10).join(p.stderr.splitlines()[-10:]))
assert p.returncode==0, f'rc={p.returncode}'

## 6. Verify invariants + FAISS ntotal (index reloaded)

In [ ]:
import json, glob, numpy as np, polars as pl, faiss
FP='dda38161938e829f2c2fc9b73d40d6cf922a5470c3b45bf176f742ee0ca7c667'
root=f'artifacts/retrieval/{FP}/m3_three_split/dense'
run_dir=sorted(glob.glob('artifacts/runs/*m3_three_split_dense*'))[-1]
meta=json.load(open(f'{run_dir}/metadata.json')); idx=json.load(open(f'{root}/index/index_metadata.json'))
keys=pl.read_parquet(f'{root}/index/product_keys.parquet')
emb=np.load(f'{root}/index/product_embeddings.npy', mmap_mode='r')
cand=pl.read_parquet(f'{root}/raw_candidates.parquet')
cat_n=pl.read_parquet(f'artifacts/datasets/esci/processed/{FP}/catalog.parquet').height
fi=faiss.read_index(f'{root}/index/faiss.index')
splits=cand.group_by('split').agg(pl.col('query_key').n_unique().alias('q')).sort('split').to_dicts()
checks={'git_commit':meta['git_commit'],'git_dirty':meta['git_dirty'],'dataset_fingerprint':meta.get('dataset_fingerprint'),
 'status':meta['status'],'device':idx['device'],'model':idx['model_name'],'revision':idx['model_revision'],
 'fine_tuned':idx['fine_tuned'],'embedding_dim':emb.shape[1],'emb_rows':emb.shape[0],'keys_rows':keys.height,
 'catalog_rows':cat_n,'counts_equal':emb.shape[0]==keys.height==cat_n==1215854,'faiss_ntotal':fi.ntotal,
 'ntotal_ok':fi.ntotal==1215854,'any_nan_5k':bool(np.isnan(np.asarray(emb[:5000])).any()),
 'cand_schema':cand.columns,'splits':splits}
print(json.dumps(checks, indent=2, default=str))
assert checks['counts_equal'] and checks['ntotal_ok'] and not checks['any_nan_5k']
assert [s['split'] for s in splits]==['test','train','validation']
print('metrics:', open(f'{root}/metrics.json').read()[:700])

## 7. DURABLY SAVE INDEX TO DRIVE + verify the durable copy
Copies embeddings, faiss.index, product_keys, index_metadata; writes checksum manifest + provenance; then reloads the Drive faiss.index to confirm ntotal. **Hard gate before any CE work.**

In [ ]:
import hashlib, json, os, shutil, glob, faiss, numpy as np, polars as pl
FP='dda38161938e829f2c2fc9b73d40d6cf922a5470c3b45bf176f742ee0ca7c667'
root=f'artifacts/retrieval/{FP}/m3_three_split/dense'; run_dir=sorted(glob.glob('artifacts/runs/*m3_three_split_dense*'))[-1]
dst='/content/drive/MyDrive/adaptirank/m3_dense_index'; os.makedirs(dst, exist_ok=True)
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda: f.read(1<<20), b''): h.update(b)
    return h.hexdigest()
shas={}
for fn in ['product_embeddings.npy','faiss.index','product_keys.parquet','index_metadata.json']:
    src=f'{root}/index/{fn}'; shutil.copy2(src, f'{dst}/{fn}'); shas[fn]=sha256(src)
    print('saved', fn, round(os.path.getsize(src)/1e6,1),'MB')
meta=json.load(open(f'{run_dir}/metadata.json'))
prov={'git_commit':meta['git_commit'],'git_dirty':meta['git_dirty'],'dataset_fingerprint':FP,'run_id':meta['run_id'],
  'artifact_shas':shas,'gpu':__import__('subprocess').run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True).stdout.strip()}
json.dump(prov, open(f'{dst}/provenance.json','w'), indent=2)
shutil.copy2(f'{run_dir}/environment.txt', f'{dst}/environment.txt')
# VERIFY durable copy: files exist, sizes match, reload faiss ntotal, checksums recompute
ok=all(os.path.isfile(f'{dst}/{fn}') for fn in shas)
fi2=faiss.read_index(f'{dst}/faiss.index'); emb2=np.load(f'{dst}/product_embeddings.npy',mmap_mode='r')
keys2=pl.read_parquet(f'{dst}/product_keys.parquet')
recheck={fn:(sha256(f'{dst}/{fn}')==shas[fn]) for fn in shas}
print('durable files present:', ok)
print('durable faiss ntotal:', fi2.ntotal, '| emb rows:', emb2.shape[0], '| keys rows:', keys2.height)
print('durable checksums match:', recheck)
assert ok and fi2.ntotal==1215854 and emb2.shape[0]==keys2.height==1215854 and all(recheck.values())
print('DURABLE INDEX VERIFIED on Drive/adaptirank/m3_dense_index/'); print(json.dumps(prov, indent=2))

## 8. Package dense candidates for transfer back

In [ ]:
import subprocess, glob
FP='dda38161938e829f2c2fc9b73d40d6cf922a5470c3b45bf176f742ee0ca7c667'
root=f'artifacts/retrieval/{FP}/m3_three_split/dense'; run_dir=sorted(glob.glob('artifacts/runs/*m3_three_split_dense*'))[-1]
out='/content/drive/MyDrive/adaptirank/m3_dense_candidates.tar.gz'
subprocess.run(['tar','--exclude=*/product_embeddings.npy','--exclude=*/faiss.index','-czf',out, root, run_dir], check=True)
print('wrote', out); subprocess.run(['ls','-lh',out])
print('\n>>> Keep this Colab session ALIVE — cross-encoder cells may follow if ready.')